# Historic Jeddah VLM Dataset — Qwen3-VL-2B Historical Buildings Caption Generation

This notebook automatically identifies eligible Historic Jeddah buildings by intersecting **building metadata** with the **human survey CSV**. A building is included only when it has a non-empty **Historical Description** in the metadata and a matching **Describe <Building Name>** column in the Survey CSV.

### Workflow
1. Install dependencies and connect Google Drive.
2. Configure the dataset, model, and output paths.
3. Load the metadata and Survey CSV.
4. Automatically identify buildings with both Historical Description and survey observations.
5. Build the metadata and human-observation context.
6. Build image-caption prompts.
7. Discover Day/Night images for every eligible building.
8. Resume from the JSONL checkpoint and generate missing captions.
9. Preview and validate the resulting JSONL dataset.



> **Model:** Qwen3-VL-2B-Instruct  
> **Inference:** Local Hugging Face Transformers inference  
> **Quantization:** 4-bit  
> **Checkpointing:** One JSONL record is saved immediately after each successful image  
> **Gemini API:** Not used


In [ ]:
import sys

!pip install -q -U \
    "transformers>=4.57.0" \
    accelerate \
    bitsandbytes \
    qwen-vl-utils \
    pandas \
    "pillow==9.5.0" \
    "scikit-image==0.19.3" \
    tqdm

# Restart kernel to ensure all packages are loaded correctly
# This is important after installing or downgrading core libraries like Pillow
if 'IPython' in sys.modules:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.kernel.do_shutdown(restart=True)

In [ ]:
pip install -U bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig

# ============================================================
# LOCAL QWEN3-VL MODEL (FIXED FOR 4-BIT QUANTIZATION)
# ============================================================

MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU was not detected. "
        "In Google Colab, enable a GPU runtime before loading Qwen3-VL."
    )

GPU_NAME = torch.cuda.get_device_name(0)

print(f"GPU: {GPU_NAME}")
print(f"CUDA: {torch.version.cuda}")

# 1. Define the 4-bit quantization configuration using BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# 2. Pass quantization_config instead of load_in_4bit=True
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=quantization_config,
)

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

print()
print("Qwen3-VL initialized successfully.")
print(f"Model: {MODEL_NAME}")
print("Quantization: 4-bit (via BitsAndBytesConfig)")

In [ ]:
import os
import re
import json
import time
import random
from pathlib import Path
from datetime import datetime

import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm.auto import tqdm

from google.genai import types

# 1. Configuration


In [ ]:
# ============================================================
# FILE PATHS
# ============================================================

DATASET_ROOT = Path(
    "/content/drive/MyDrive/processed_dataset"
)

SURVEY_CSV = Path(
    "/content/drive/MyDrive/merged_surveys_output.csv"
)

METADATA_CSV = Path(
    "/content/drive/MyDrive/processed_dataset/"
    "MDT-001_Building_Metadata_Template(Building Metadata).csv"
)

# NEW OUTPUT FILE
# This is intentionally separate from the Gemini JSONL.
OUTPUT_JSONL_PATH = (
    DATASET_ROOT
    / "qwen3_vl_2b_albalad_test_dataset_v1.jsonl"
)

FAILED_JSONL_PATH = (
    DATASET_ROOT
    / "failed_qwen3_vl_2b_test_images_v1.jsonl"
)

RUN_LOG_PATH = (
    DATASET_ROOT
    / "qwen3_vl_2b_test_run_v1.log"
)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"

# The model is loaded locally in Section 0 using 4-bit
# quantization for Colab GPU memory efficiency.


# ============================================================
# TESTING CONFIGURATION
# ============================================================

# Number of images to test from each building.
#
# Set to None to process every discovered image.
TEST_LIMIT_PER_HOUSE = None

# False = resume from the existing Qwen3-VL JSONL checkpoint.
# True  = regenerate every selected image.
#
# IMPORTANT:
# If you want a completely fresh 1,306-image dataset,
# set this to True AND use the new output filename above.
OVERWRITE_EXISTING = True


# ============================================================
# LOCAL INFERENCE CONFIGURATION
# ============================================================

# Local inference does not consume Gemini API quota.
MAX_RETRIES = 3
INITIAL_BACKOFF_SECONDS = 5
MAX_BACKOFF_SECONDS = 60

# No API rate-limit delay is required.
# A small delay can be useful if the GPU is thermally constrained.
REQUEST_DELAY_SECONDS = 0.2


# ============================================================
# GENERATION CONFIGURATION
# ============================================================

MAX_OUTPUT_TOKENS = 280
TEMPERATURE = 0.2
TOP_P = 0.9

MIN_CAPTION_WORDS = 20

SUPPORTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}


## 1.1 Automatic building selection

The house list is generated automatically after the metadata and Survey CSV are loaded.


In [ ]:
# This variable is populated automatically in Section 2.1.
# It is intentionally kept empty here so the notebook has no hard-coded house list.
HOUSE_CONFIG = {}

## 1.2 Validate inputs

Check that the required CSV files and building image folders are available before processing.


In [ ]:
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

required_files = {
    "Survey CSV": SURVEY_CSV,
    "Metadata CSV": METADATA_CSV,
}

missing_files = [
    (label, path)
    for label, path in required_files.items()
    if not path.exists()
]

if missing_files:
    error_message = "\n".join(
        f"{label}: {path}"
        for label, path in missing_files
    )
    raise FileNotFoundError(
        "The following required files were not found:\n"
        f"{error_message}"
    )

print("Survey CSV found:")
print(f"  {SURVEY_CSV}")
print("\nMetadata CSV found:")
print(f"  {METADATA_CSV}")


# 2. Building metadata


In [ ]:
metadata_df = pd.read_csv(
    METADATA_CSV,
    encoding="utf-8-sig",
)

# Remove completely empty columns such as "Unnamed: 18".
metadata_df = metadata_df.dropna(
    axis=1,
    how="all",
)

# Clean column names.
metadata_df.columns = [
    str(column).strip()
    for column in metadata_df.columns
]

print("Metadata columns:")
print(metadata_df.columns.tolist())

print(f"\nTotal metadata rows: {len(metadata_df)}")

## 2.1 Automatically identify eligible buildings

A building is eligible when **both** conditions are satisfied:
- `Historical Description` in the metadata CSV is populated.
- The Survey CSV contains a matching `Describe <Building Name (English)>` column.

The building ID and image-folder path are then generated from the metadata table.


In [ ]:
survey_df = pd.read_csv(
    SURVEY_CSV,
    encoding="utf-8-sig",
)

print(f"Survey rows: {len(survey_df)}")

In [ ]:
import pandas as pd
import re
from pathlib import Path
from difflib import SequenceMatcher


def has_value(value):
    """Return True when a CSV cell contains meaningful content."""
    if pd.isna(value):
        return False


    text = str(value).strip()


    if not text:
        return False


    return text.lower() not in {
        "nan", "none", "null", "n/a", "na",
        "not provided", "not available", "unknown",
    }


def clean_for_matching(text):
    """Clean and normalize names/column labels for fuzzy matching."""
    if pd.isna(text):
        return ""
    text = str(text).lower()


    # Remove common filler words and numbers (e.g., describe, bayt, house, 1, 2)
    text = re.sub(r'\b(describe|bayt|house|al|ribat|waqf|mosque|1|2|3|4|5|6|7|8|9|10|-)\b', ' ', text)
    # Remove non-alphabetical characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Handle known extreme transliteration variations
    text = text.replace('xenel', 'zainal')


    # Collapse whitespace
    return re.sub(r'\s+', ' ', text).strip()


def similar(a, b):
    """Calculate string similarity ratio."""
    return SequenceMatcher(None, a, b).ratio()




# Clean metadata building IDs and names before matching.
metadata_df["Building ID"] = (
    metadata_df["Building ID"]
    .astype(str)
    .str.strip()
)


metadata_df["Building Name (English)"] = (
    metadata_df["Building Name (English)"]
    .astype(str)
    .str.strip()
)


# Map cleaned survey columns to their original exact column names
survey_columns = {}
for column in survey_df.columns:
    cleaned_col = clean_for_matching(column)
    if cleaned_col: # Ignore empty strings like 'Timestamp' after cleaning
        survey_columns[cleaned_col] = str(column).strip()


eligible_rows = []
selection_report = []


for _, row in metadata_df.iterrows():
    building_id = row["Building ID"]
    building_name = row["Building Name (English)"]
    historical_description = row.get("Historical Description")


    # Clean the metadata building name
    c_name = clean_for_matching(building_name)


    # Fuzzy match the building name against the survey columns
    best_match_orig_col = None
    best_score = 0


    for cleaned_col, orig_col in survey_columns.items():
        # Split the cleaned metadata name into individual unique words
        words = c_name.split()
        score = 0
        if words:
            # Check maximum similarity of any word against the cleaned survey column
            score = max(similar(w, cleaned_col) for w in words)


        if score > best_score:
            best_score = score
            best_match_orig_col = orig_col


    has_history = has_value(historical_description)


    # We require a strong match score (> 70%) to avoid false positives
    has_survey_column = (best_score > 0.70)


    if has_history and has_survey_column:
        row = row.copy()
        row["Survey Column"] = best_match_orig_col
        eligible_rows.append(row)


        selection_report.append({
            "Building ID": building_id,
            "Building Name": building_name,
            "Historical Description": "Yes",
            "Survey Column": best_match_orig_col,
            "Match Confidence": f"{best_score:.0%}",
            "Image Folder": str(DATASET_ROOT / building_id),
        })


# Convert eligible rows to a DataFrame.
selected_metadata_df = pd.DataFrame(eligible_rows)


# Build the runtime configuration automatically.
HOUSE_CONFIG = {}


if not selected_metadata_df.empty:
    for _, row in selected_metadata_df.iterrows():
        building_id = row["Building ID"]
        matched_column = row["Survey Column"]


        HOUSE_CONFIG[building_id] = {
            "path": DATASET_ROOT / building_id,
            "survey_column": matched_column,
        }


SELECTED_HOUSE_IDS = list(HOUSE_CONFIG.keys())


print(f"\nEligible buildings: {len(SELECTED_HOUSE_IDS)}")
if selection_report:
    display(pd.DataFrame(selection_report))


# Report folders that are missing without stopping the metadata/survey selection.
missing_image_folders = [
    (building_id, config["path"])
    for building_id, config in HOUSE_CONFIG.items()
    if not Path(config["path"]).exists()
]


if missing_image_folders:
    print("\nWarning: image folders are missing for:")
    for building_id, path in missing_image_folders:
        print(f"- {building_id}: {path}")
else:
    print("\nAll eligible building image folders were found.")


print("\nSelected building IDs:")
for building_id in SELECTED_HOUSE_IDS:
    print(f"- {building_id}")

## 2.2 Clean metadata values

Normalize missing values, whitespace, and floor-number formatting before building prompts.


In [ ]:
def clean_text_value(value, default="Not provided"):
    """
    Convert a CSV value into clean plain text.
    """

    if pd.isna(value):
        return default

    text = str(value).strip()

    if not text:
        return default

    if text.lower() in {
        "nan",
        "none",
        "null",
        "n/a",
        "na",
    }:
        return default

    # Replace repeated whitespace and multiline material lists.
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def clean_number_of_floors(value):
    """
    Convert floor values such as 4.0 into '4'.
    """

    if pd.isna(value):
        return "Not provided"

    try:
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(int(numeric_value))

        return str(numeric_value)

    except (TypeError, ValueError):
        return clean_text_value(value)

## 2.3 Build metadata lookup

Create a dictionary keyed by building ID for fast prompt construction.


In [ ]:
building_metadata = {}

for _, row in selected_metadata_df.iterrows():
    building_id = clean_text_value(
        row.get("Building ID"),
        default="",
    )

    building_metadata[building_id] = {
        "building_id": building_id,

        "english_name": clean_text_value(
            row.get("Building Name (English)")
        ),

        "arabic_name": clean_text_value(
            row.get("Building Name (Arabic)")
        ),

        "district": clean_text_value(
            row.get("District")
        ),

        "street": clean_text_value(
            row.get("Street")
        ),

        "area": clean_text_value(
            row.get("Area")
        ),

        "city": clean_text_value(
            row.get("City")
        ),

        "gps_coordinates": clean_text_value(
            row.get("GPS Coordinates")
        ),

        "category": clean_text_value(
            row.get("Building Category")
        ),

        "architectural_style": clean_text_value(
            row.get("Architectural Style")
        ),

        "construction_material": clean_text_value(
            row.get("Construction Material")
        ),

        "number_of_floors": clean_number_of_floors(
            row.get("Number of Floors")
        ),

        "architectural_elements": clean_text_value(
            row.get("Architectural Elements")
        ),

        "metadata_lighting": clean_text_value(
            row.get("Lighting Condition")
        ),

        "historical_description": clean_text_value(
            row.get("Historical Description")
        ),

        "reference_source": clean_text_value(
            row.get("Reference Source")
        ),

        "historical_source": clean_text_value(
            row.get("Source of Historical Description")
        ),
    }

print(
    f"Loaded metadata for "
    f"{len(building_metadata)} selected buildings."
)

for building_id, metadata in building_metadata.items():
    print(
        f"- {building_id}: "
        f"{metadata['english_name']} / "
        f"{metadata['arabic_name']}"
    )

# 3. Human survey context

Use the Survey CSV responses associated with the automatically selected buildings.


In [ ]:
# The Survey CSV was loaded in Section 2.1 because it determines eligibility.
# Reuse the same dataframe here rather than loading it a second time.

print("Survey columns:")
for column in survey_df.columns:
    print(f"- {column}")


## 3.1 Clean and combine survey responses
Clean and combine duplicate human observations for each building.


In [ ]:
def clean_survey_response(value):
    """
    Clean one human survey response.
    """

    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    if text.lower() in {
        "nan",
        "none",
        "null",
        "n/a",
        "na",
    }:
        return None

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def load_survey_descriptions(
    dataframe,
    house_config,
):
    """
    Create one combined survey context for each building.
    """

    survey_lookup = {}

    for house_id, config in house_config.items():
        column_name = config["survey_column"]

        if column_name not in dataframe.columns:
            print(
                f"Warning: Survey column was not found: "
                f"{column_name}"
            )

            survey_lookup[house_id] = (
                "No human survey observations were provided."
            )
            continue

        responses = []

        for value in dataframe[column_name]:
            cleaned_response = clean_survey_response(value)

            if cleaned_response:
                responses.append(cleaned_response)

        # Remove duplicate responses while preserving order.
        unique_responses = list(dict.fromkeys(responses))

        if unique_responses:
            survey_lookup[house_id] = "\n".join(
                f"- {response}"
                for response in unique_responses
            )
        else:
            survey_lookup[house_id] = (
                "No human survey observations were provided."
            )

        print(
            f"{house_id}: "
            f"{len(unique_responses)} unique survey responses"
        )

    return survey_lookup


house_survey_descriptions = load_survey_descriptions(
    survey_df,
    HOUSE_CONFIG,
)

## 3.2 Preview combined context
Inspect the metadata and survey observations that will be supplied to the vision-language model.


In [ ]:
for house_id in SELECTED_HOUSE_IDS:
    metadata = building_metadata[house_id]

    print("=" * 80)
    print(
        f"{metadata['english_name']} "
        f"({metadata['arabic_name']})"
    )
    print("=" * 80)

    print("\nMetadata:")
    print(
        json.dumps(
            metadata,
            ensure_ascii=False,
            indent=2,
        )
    )

    print("\nSurvey observations:")
    print(house_survey_descriptions[house_id])

    print()

# 4. Caption generation prompt


In [ ]:
PROMPT_TEMPLATE = """
You are an expert local tour guide in Historic Jeddah (Al-Balad), known for your deep knowledge of the district's heritage houses and your ability to describe them vividly to visitors standing right in front of them. You are looking at a specific photograph of a historic building and will describe it exactly as you would to a group of visitors on a walking tour — precise, engaging, and grounded only in what is visible.

### CONTEXTUAL METADATA
Building ID: {building_id}
Name: {english_name} ({arabic_name})
Location: {street}, {district}, {area}, {city}

### REFERENCE ATTRIBUTES (Vocabulary Guide)
Category: {category}
Style: {architectural_style}
Materials: {construction_material}
Floors: {number_of_floors}
Key Elements: {architectural_elements}
Lighting Condition: {lighting}

### HUMAN OBSERVATIONS
{human_descriptions}

### EXECUTION RULES
1. VISUAL PRIMACY: The image is your absolute source of truth. Use the "Reference Attributes" and "Human Observations" strictly as a vocabulary guide to correctly name the features you see. If a human observation conflicts with what the image actually shows, trust the image.
2. NO HALLUCINATIONS: Do not mention historical dates, events, hidden floors, or unseen materials. If a feature (like a Roshan window or coral stone) is not clearly visible in this specific lighting and angle, do not describe it.
3. NAME THE BUILDING: Naturally mention the building name ({english_name}) exactly once within the description, the way a tour guide would when first pointing it out to a group. Do not repeat the name more than once.
4. DYNAMIC SENTENCE STRUCTURE (CRITICAL): Do NOT start your description with formulaic phrases like "Captured from an upward-looking perspective..." or "This image shows...". Vary your opening sentences as a guide naturally would — sometimes drawing attention to a striking feature first, sometimes setting the scene, sometimes starting with the woodwork or masonry.
5. INTEGRATED CONTEXT: Weave the viewing angle (e.g., street-level, upward-looking, oblique) and the lighting condition naturally into your commentary rather than forcing them into the opening sentence.
6. ARCHITECTURAL FOCUS: Focus heavily on visible characteristics (e.g., rawashin, mashrabiya, shish windows, decorative plaster, woodwork, masonry) and surrounding visible streetscape elements (e.g., pavements, trees, adjacent structures) — the details a guide would point out to make visitors look closer.
7. CONCISENESS: The description must be exactly 3 to 4 complete, professionally written sentences, in a warm but informative guide's voice. Do not use introductory filler, lists, or markdown formatting.
8. STRICT JSON OUTPUT: Return only a valid JSON object in the exact format below. No text before or after it.

OUTPUT FORMAT:
{{
  "architectural_description": "Your 3 to 4 sentence description goes here."
}}
""".strip()

## 4.1 Build prompts

Combine building metadata, human observations, and image lighting context into the final model prompt.


In [ ]:
def build_prompt(task):
    """
    Combine metadata, survey observations and image context.
    """

    house_id = task["house_id"]
    metadata = building_metadata[house_id]

    return PROMPT_TEMPLATE.format(
        building_id=metadata["building_id"],
        english_name=metadata["english_name"],
        arabic_name=metadata["arabic_name"],
        district=metadata["district"],
        street=metadata["street"],
        area=metadata["area"],
        city=metadata["city"],
        category=metadata["category"],
        architectural_style=metadata[
            "architectural_style"
        ],
        construction_material=metadata[
            "construction_material"
        ],
        number_of_floors=metadata[
            "number_of_floors"
        ],
        architectural_elements=metadata[
            "architectural_elements"
        ],
        historical_description=metadata[
            "historical_description"
        ],
        human_descriptions=house_survey_descriptions[
            house_id
        ],
        lighting=task["lighting"],
    )

# 5. Image discovery and task selection


In [ ]:
from pathlib import Path
from typing import List, Dict, Any

def infer_lighting(image_path: Path, house_directory: Path) -> str:
    """
    Infer lighting from parent folder names using specific suffixes.
    """
    # Extract parts of the path relative to the house directory, excluding the filename
    relative_parts = image_path.relative_to(house_directory).parts[:-1]

    for part in relative_parts:
        # Convert to lowercase once for safe comparison
        part_lower = part.lower()

        # Match 'day', 'night' or lowercase suffixes '_d', '_n'
        if "day" in part_lower or part_lower.endswith("_d"):
            return "Day"
        if "night" in part_lower or part_lower.endswith("_n"):
            return "Night"

    return "Unknown"


def create_relative_path(image_path: Path) -> str:
    """
    Create a portable path relative to DATASET_ROOT.
    """
    return image_path.relative_to(DATASET_ROOT).as_posix()


def discover_house_images(house_id: str, config: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Recursively discover supported images for one house and map them to processing tasks.
    """
    house_directory = Path(config["path"])
    tasks = []

    if not house_directory.exists():
        print(f"Warning: Folder not found for {house_id}: {house_directory}")
        return tasks

    # Use a generator expression inside sorted() for memory efficiency
    image_files = sorted(
        file_path
        for file_path in house_directory.rglob("*")
        if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_EXTENSIONS
    )

    for image_path in image_files:
        tasks.append({
            "absolute_path": image_path,
            "relative_path": create_relative_path(image_path),
            "house_id": house_id,
            "lighting": infer_lighting(image_path, house_directory),
        })

    return tasks

## 5.1 Build the test set

Discover images recursively for every automatically selected building. When a per-building limit is used, keep Day/Night coverage balanced where possible.


In [ ]:
all_tasks = []
selected_tasks = []

for house_id, config in HOUSE_CONFIG.items():
    house_tasks = discover_house_images(
        house_id,
        config,
    )

    all_tasks.extend(house_tasks)

    if TEST_LIMIT_PER_HOUSE is None:
        limited_tasks = house_tasks
    else:
        # 1. Partition tasks by lighting condition
        day_tasks = [t for t in house_tasks if t["lighting"] == "Day"]
        night_tasks = [t for t in house_tasks if t["lighting"] == "Night"]

        # 2. Calculate the split limit (e.g., 10 // 2 = 5)
        half_limit = TEST_LIMIT_PER_HOUSE // 2

        # 3. Combine the sliced lists
        limited_tasks = (
            day_tasks[:half_limit] +
            night_tasks[:half_limit]
        )

    selected_tasks.extend(limited_tasks)

    metadata = building_metadata[house_id]

    print(
        f"{metadata['english_name']}: "
        f"{len(house_tasks)} images found, "
        f"{len(limited_tasks)} selected for this run."
    )

selected_tasks.sort(
    key=lambda task: (
        task["house_id"],
        task["lighting"],
        task["relative_path"],
    )
)

print(f"\nTotal discovered images: {len(all_tasks)}")
print(f"Images selected for testing: {len(selected_tasks)}")

## 5.2 Review selected images

Review the complete task list before running local Qwen3-VL inference.


In [ ]:
task_preview = []

for task in selected_tasks:
    metadata = building_metadata[
        task["house_id"]
    ]

    task_preview.append(
        {
            "building_id": task["house_id"],
            "building_name": metadata[
                "english_name"
            ],
            "lighting": task["lighting"],
            "image": task["relative_path"],
        }
    )

task_preview_df = pd.DataFrame(task_preview)

display(task_preview_df)

## 5.3 Check image integrity and dimensions

Verify that selected images can be opened and record their dimensions.


In [ ]:
def inspect_image_sizes(tasks):
    """
    Count dimensions and identify unreadable images.
    """

    size_counts = {}
    invalid_images = []

    for task in tqdm(
        tasks,
        desc="Checking images",
    ):
        try:
            with Image.open(
                task["absolute_path"]
            ) as image:
                size = image.size

            size_counts[size] = (
                size_counts.get(size, 0) + 1
            )

        except Exception as error:
            invalid_images.append(
                {
                    "image": task["relative_path"],
                    "error": str(error),
                }
            )

    return size_counts, invalid_images


size_counts, invalid_images = inspect_image_sizes(
    selected_tasks
)

print("Image dimensions:")

for size, count in sorted(
    size_counts.items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(
        f"{size[0]} × {size[1]}: "
        f"{count} images"
    )

if invalid_images:
    print(
        f"\nUnreadable images: "
        f"{len(invalid_images)}"
    )

    display(pd.DataFrame(invalid_images))
else:
    print("\nAll selected images are readable.")

## 5.4 Image loading

Open images safely, apply EXIF orientation, and convert unsupported color modes to RGB.


In [ ]:
def load_and_validate_image(image_path):
    """
    Validate and load an image without resizing it.
    """

    # Verify file integrity.
    with Image.open(image_path) as image:
        image.verify()

    # Reopen after verify().
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image)

        if image.mode not in {
            "RGB",
            "RGBA",
        }:
            image = image.convert("RGB")

        image_copy = image.copy()

    return image_copy

# 6. Checkpointing and output


In [ ]:
def load_processed_images(jsonl_path):
    """
    Read successfully processed image paths from JSONL.
    """

    processed = set()
    malformed_lines = []

    if not jsonl_path.exists():
        return processed, malformed_lines

    with jsonl_path.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

                image_path = record.get("image")

                if image_path:
                    processed.add(image_path)
                else:
                    malformed_lines.append(
                        line_number
                    )

            except json.JSONDecodeError:
                malformed_lines.append(
                    line_number
                )

    return processed, malformed_lines


processed_images, malformed_lines = (
    load_processed_images(
        OUTPUT_JSONL_PATH
    )
)

print(
    f"Previously completed images: "
    f"{len(processed_images)}"
)

if malformed_lines:
    print(
        "Malformed output lines:",
        malformed_lines,
    )

## 6.1 Resume unfinished work

Read completed image paths from the JSONL checkpoint and skip them unless overwrite mode is enabled.


In [ ]:
if OVERWRITE_EXISTING:
    pending_tasks = selected_tasks.copy()
else:
    pending_tasks = [
        task
        for task in selected_tasks
        if task["relative_path"]
        not in processed_images
    ]

print(
    f"Selected test images: "
    f"{len(selected_tasks)}"
)

print(
    f"Already completed: "
    f"{len(selected_tasks) - len(pending_tasks)}"
)

print(
    f"Remaining images: "
    f"{len(pending_tasks)}"
)

## 6.2 Caption cleaning, validation, and checkpoint utilities
These helpers normalize model output and persist results safely during long-running runs.


In [ ]:
def clean_caption(text):
    """
    Normalize Qwen3-VL output and extract only the
    architectural_description field when JSON is returned.

    Qwen may return:
    1. valid JSON
    2. JSON inside markdown fences
    3. plain text
    """

    if text is None:
        return ""

    caption = str(text).strip()

    # --------------------------------------------------------
    # Remove markdown code fences.
    # --------------------------------------------------------
    caption = re.sub(
        r"^```(?:json|text|markdown)?\s*",
        "",
        caption,
        flags=re.IGNORECASE,
    )

    caption = re.sub(
        r"\s*```$",
        "",
        caption,
    ).strip()

    # --------------------------------------------------------
    # Try to parse the required JSON object.
    # --------------------------------------------------------
    try:
        parsed = json.loads(caption)

        if isinstance(parsed, dict):
            description = parsed.get(
                "architectural_description"
            )

            if description:
                caption = str(description).strip()

    except json.JSONDecodeError:
        # Qwen sometimes adds a short prefix/suffix around JSON.
        # Try to recover the JSON object from the response.
        json_match = re.search(
            r"\{.*\}",
            caption,
            flags=re.DOTALL,
        )

        if json_match:
            try:
                parsed = json.loads(
                    json_match.group(0)
                )

                if isinstance(parsed, dict):
                    description = parsed.get(
                        "architectural_description"
                    )

                    if description:
                        caption = str(description).strip()

            except json.JSONDecodeError:
                pass

    # --------------------------------------------------------
    # Remove accidental headings.
    # --------------------------------------------------------
    caption = re.sub(
        r"^(description|architectural description)"
        r"\s*:\s*",
        "",
        caption,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # Remove an accidental leading bullet.
    # --------------------------------------------------------
    caption = re.sub(
        r"^\s*[-*]\s+",
        "",
        caption,
    )

    # --------------------------------------------------------
    # Normalize whitespace.
    # --------------------------------------------------------
    caption = re.sub(
        r"[ \t]+",
        " ",
        caption,
    )

    caption = re.sub(
        r"\n+",
        " ",
        caption,
    )

    return caption.strip()


def validate_caption(caption):
    """
    Check that the generated caption is non-empty,
    sufficiently long, and approximately follows the
    requested 3-4 sentence structure.
    """

    if not caption:
        return False, "Caption is empty."

    word_count = len(caption.split())

    if word_count < MIN_CAPTION_WORDS:
        return (
            False,
            f"Caption contains only {word_count} words.",
        )

    # Count sentence-ending punctuation.
    sentence_count = len(
        re.findall(
            r"[.!?](?:\s|$)",
            caption,
        )
    )

    # Be tolerant of model punctuation while still checking
    # the intended 3-4 sentence format.
    if sentence_count not in {3, 4}:
        return (
            False,
            f"Expected 3-4 sentences; detected "
            f"{sentence_count}.",
        )

    return True, None


In [ ]:
def append_jsonl_record(
    file_path,
    record,
):
    """
    Append one JSON object immediately.
    """

    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    serialized = json.dumps(
        record,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    with file_path.open(
        "a",
        encoding="utf-8",
    ) as file:
        file.write(serialized + "\n")
        file.flush()
        os.fsync(file.fileno())


def append_run_log(message):
    """
    Append a timestamped text log.
    """

    timestamp = datetime.now().isoformat(
        timespec="seconds"
    )

    with RUN_LOG_PATH.open(
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            f"[{timestamp}] {message}\n"
        )


def log_failed_image(
    task,
    error,
    attempts,
):
    """
    Save a permanently failed image.
    """

    failure_record = {
        "image": task["relative_path"],
        "house_id": task["house_id"],
        "lighting": task["lighting"],
        "attempts": attempts,
        "error_type": type(error).__name__,
        "error": str(error),
        "timestamp": datetime.now().isoformat(
            timespec="seconds"
        ),
    }

    append_jsonl_record(
        FAILED_JSONL_PATH,
        failure_record,
    )

In [ ]:
def generate_caption_with_retry(task):
    """
    Generate a caption with local Qwen3-VL inference.

    No external API is used.
    """

    prompt = build_prompt(task)

    image = load_and_validate_image(
        task["absolute_path"]
    )

    last_error = None

    try:
        for attempt in range(
            1,
            MAX_RETRIES + 1,
        ):
            try:
                # Qwen3-VL expects a multimodal chat message.
                messages = [
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "image": image,
                            },
                            {
                                "type": "text",
                                "text": prompt,
                            },
                        ],
                    }
                ]

                # Convert the message into Qwen3-VL model inputs.
                inputs = processor.apply_chat_template(
                    messages,
                    tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True,
                    return_tensors="pt",
                )

                # Move tensors to the model's device.
                inputs = inputs.to(
                    model.device
                )

                with torch.inference_mode():
                    generated_ids = model.generate(
                        **inputs,
                        max_new_tokens=MAX_OUTPUT_TOKENS,
                        do_sample=True,
                        temperature=TEMPERATURE,
                        top_p=TOP_P,
                    )

                # Remove the prompt tokens so only the newly
                # generated answer is decoded.
                generated_ids_trimmed = [
                    output_ids[len(input_ids):]
                    for input_ids, output_ids
                    in zip(
                        inputs.input_ids,
                        generated_ids,
                    )
                ]

                response_text = processor.batch_decode(
                    generated_ids_trimmed,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False,
                )[0]

                caption = clean_caption(
                    response_text
                )

                is_valid, validation_error = (
                    validate_caption(caption)
                )

                if not is_valid:
                    raise ValueError(
                        validation_error
                    )

                # Release inference tensors before the next image.
                del inputs
                del generated_ids
                del generated_ids_trimmed

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                return caption, attempt

            except Exception as error:
                last_error = error

                if attempt >= MAX_RETRIES:
                    break

                # Clear GPU memory before retrying.
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                exponential_delay = min(
                    INITIAL_BACKOFF_SECONDS
                    * (2 ** (attempt - 1)),
                    MAX_BACKOFF_SECONDS,
                )

                jitter = random.uniform(0, 2)

                wait_seconds = (
                    exponential_delay + jitter
                )

                tqdm.write(
                    f"\nAttempt "
                    f"{attempt}/{MAX_RETRIES} "
                    f"failed for "
                    f"{task['relative_path']}\n"
                    f"{type(error).__name__}: "
                    f"{error}\n"
                    f"Retrying in "
                    f"{wait_seconds:.1f} seconds."
                )

                time.sleep(wait_seconds)

    finally:
        image.close()

    raise RuntimeError(
        f"Caption generation failed after "
        f"{MAX_RETRIES} attempts. "
        f"Last error: {last_error}"
    )


In [ ]:
def create_dataset_record(
    task,
    caption,
):
    """
    Create one Qwen-style VLM conversation record.
    """

    metadata = building_metadata[
        task["house_id"]
    ]

    if task["lighting"] == "Unknown":
        user_instruction = (
            "<image>\n"
            "Describe this historic building "
            "in Al-Balad."
        )
    else:
        user_instruction = (
            "<image>\n"
            "Describe this historic building "
            "in Al-Balad during the "
            f"{task['lighting'].lower()}."
        )

    return {
        "image": task["relative_path"],

        "metadata": {
            "building_id": task["house_id"],
            "building_name_english": metadata[
                "english_name"
            ],
            "building_name_arabic": metadata[
                "arabic_name"
            ],
            "lighting": task["lighting"],
            "model": MODEL_NAME,
            "image_width": 224,
            "image_height": 224,
            "caption_source": (
                "Qwen3-VL-2B-Instruct with building "
                "metadata and human survey context"
            ),
        },

        "conversations": [
            {
                "role": "user",
                "content": user_instruction,
            },
            {
                "role": "assistant",
                "content": caption,
            },
        ],
    }

## 6.3 Preview one pending image and prompt

Inspect one pending image and the exact prompt that will be sent to **Qwen3-VL-2B-Instruct**.


In [ ]:
if not pending_tasks:
    print(
        "No pending test images remain."
    )

else:
    preview_task = pending_tasks[0]

    preview_metadata = building_metadata[
        preview_task["house_id"]
    ]

    print("Selected image:")
    print(preview_task["relative_path"])

    print("\nBuilding:")
    print(
        preview_metadata["english_name"]
    )

    preview_image = load_and_validate_image(
        preview_task["absolute_path"]
    )

    display(preview_image)

    preview_image.close()

    print("\n" + "=" * 80)
    print("COMPLETE PROMPT")
    print("=" * 80)
    print(build_prompt(preview_task))

## 6.4 Test one caption generation

Run one local Qwen3-VL inference before processing the full pending set.


In [ ]:
if not pending_tasks:
    print(
        "No pending test images remain."
    )

else:
    test_task = pending_tasks[0]

    print(
        f"Generating a caption for:\n"
        f"{test_task['relative_path']}\n"
    )

    test_caption, attempts_used = (
        generate_caption_with_retry(
            test_task
        )
    )

    print(
        f"Attempts used: "
        f"{attempts_used}"
    )

    print("\nGenerated caption:\n")
    print(test_caption)

## 6.5 Process the remaining images

Generate captions locally with Qwen3-VL, retry failed inference, and checkpoint every successful record immediately.


In [ ]:
def process_dataset(tasks):
    """
    Process all pending tasks sequentially.

    Each successful record is saved immediately.
    """

    successful_count = 0
    failed_count = 0
    total_attempts = 0

    started_at = time.time()

    append_run_log(
        f"Run started; "
        f"model={MODEL_NAME}; "
        f"tasks={len(tasks)}"
    )

    progress_bar = tqdm(
        tasks,
        desc="Generating captions",
        unit="image",
        dynamic_ncols=True,
    )

    for task in progress_bar:
        task_started_at = time.time()

        try:
            caption, attempts_used = (
                generate_caption_with_retry(
                    task
                )
            )

            record = create_dataset_record(
                task,
                caption,
            )

            # Immediate checkpoint.
            append_jsonl_record(
                OUTPUT_JSONL_PATH,
                record,
            )

            successful_count += 1
            total_attempts += attempts_used

            processing_seconds = (
                time.time() - task_started_at
            )

            progress_bar.set_postfix(
                saved=successful_count,
                failed=failed_count,
                seconds=(
                    f"{processing_seconds:.1f}"
                ),
            )

        except Exception as error:
            failed_count += 1

            log_failed_image(
                task=task,
                error=error,
                attempts=MAX_RETRIES,
            )

            tqdm.write(
                f"\nPermanently failed:\n"
                f"{task['relative_path']}\n"
                f"{type(error).__name__}: "
                f"{error}\n"
            )

        # Maintain a small local inference interval.
        elapsed = time.time() - task_started_at

        remaining_delay = (
            REQUEST_DELAY_SECONDS - elapsed
        )

        if remaining_delay > 0:
            time.sleep(remaining_delay)

    total_elapsed = time.time() - started_at

    summary = {
        "successful": successful_count,
        "failed": failed_count,
        "selected_tasks": len(tasks),
        "total_attempts": total_attempts,
        "elapsed_seconds": round(
            total_elapsed,
            2,
        ),
        "average_seconds_per_task": round(
            total_elapsed / max(len(tasks), 1),
            2,
        ),
    }

    append_run_log(
        "Run completed; "
        + json.dumps(
            summary,
            ensure_ascii=False,
        )
    )

    return summary

# 7. Run caption generation

Start the full generation run for all pending images.


In [ ]:
if not pending_tasks:
    print(
        "Nothing to process. "
        "All selected test images already exist "
        "in the output JSONL."
    )

else:
    processing_summary = process_dataset(
        pending_tasks
    )

    print("\nProcessing completed.")

    print(
        json.dumps(
            processing_summary,
            ensure_ascii=False,
            indent=2,
        )
    )

    print("\nOutput:")
    print(OUTPUT_JSONL_PATH)

    print("\nFailures:")
    print(FAILED_JSONL_PATH)

# 8. Review generated dataset

Load a small preview of the JSONL output for a quick quality check.


In [ ]:
# 8. Review generated dataset

Load a small preview of the Qwen3-VL JSONL output for a quick quality check.


## 8.1 Validate JSONL structure

Check JSON syntax, required conversation structure, empty captions, and duplicate image records.


In [ ]:
def validate_dataset_jsonl(jsonl_path):
    """
    Validate syntax, required fields and duplicates.
    """

    valid_records = 0
    invalid_records = []
    duplicate_records = []

    seen_images = set()

    if not jsonl_path.exists():
        print(
            "Output JSONL does not exist."
        )

        return None

    with jsonl_path.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

                image_path = record.get("image")

                if not image_path:
                    raise ValueError(
                        "Missing image path."
                    )

                if image_path in seen_images:
                    duplicate_records.append(
                        {
                            "line": line_number,
                            "image": image_path,
                        }
                    )
                else:
                    seen_images.add(image_path)

                conversations = record.get(
                    "conversations"
                )

                if not isinstance(
                    conversations,
                    list,
                ):
                    raise ValueError(
                        "conversations is not a list."
                    )

                if len(conversations) != 2:
                    raise ValueError(
                        "Expected exactly two "
                        "conversation messages."
                    )

                if (
                    conversations[0].get("role")
                    != "user"
                ):
                    raise ValueError(
                        "First message is not user."
                    )

                if (
                    conversations[1].get("role")
                    != "assistant"
                ):
                    raise ValueError(
                        "Second message is not assistant."
                    )

                caption = conversations[1].get(
                    "content",
                    "",
                ).strip()

                if not caption:
                    raise ValueError(
                        "Assistant caption is empty."
                    )

                valid_records += 1

            except Exception as error:
                invalid_records.append(
                    {
                        "line": line_number,
                        "error": str(error),
                    }
                )

    report = {
        "valid_records": valid_records,
        "invalid_records": len(
            invalid_records
        ),
        "duplicate_records": len(
            duplicate_records
        ),
        "unique_images": len(
            seen_images
        ),
    }

    print(
        json.dumps(
            report,
            indent=2,
        )
    )

    if invalid_records:
        print("\nInvalid records:")
        display(
            pd.DataFrame(
                invalid_records[:20]
            )
        )

    if duplicate_records:
        print("\nDuplicate records:")
        display(
            pd.DataFrame(
                duplicate_records[:20]
            )
        )

    return report


validation_report = validate_dataset_jsonl(
    OUTPUT_JSONL_PATH
)

## 8.2 Confirm completion

Compare selected image paths with completed JSONL records to confirm that the test set is complete.


In [ ]:
final_processed_images, malformed_lines = (
    load_processed_images(
        OUTPUT_JSONL_PATH
    )
)

selected_paths = {
    task["relative_path"]
    for task in selected_tasks
}

completed_selected_paths = (
    selected_paths
    & final_processed_images
)

missing_selected_paths = sorted(
    selected_paths
    - final_processed_images
)

print(
    f"Selected test images: "
    f"{len(selected_paths)}"
)

print(
    f"Completed test images: "
    f"{len(completed_selected_paths)}"
)

print(
    f"Missing test images: "
    f"{len(missing_selected_paths)}"
)

print(
    f"Malformed JSONL lines: "
    f"{len(malformed_lines)}"
)

if missing_selected_paths:
    print("\nMissing images:")

    for image_path in missing_selected_paths:
        print(image_path)
else:
    print(
        "\nAll selected test images "
        "were completed successfully."
    )